# Simulación Monte Carlo para riesgo de presupuesto

Este notebook estima la distribución posible del costo total de un proyecto mediante simulación Monte Carlo y cuantifica la probabilidad de cumplir un presupuesto.

## Objetivo

- Modelar la incertidumbre de los componentes del costo.
- Ejecutar miles de escenarios reproducibles.
- Calcular P50, P80 y P90.
- Estimar la probabilidad de cumplir el presupuesto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
N_SIMULACIONES = 50_000
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 1. Supuestos del presupuesto

Para cada componente se definen estimaciones optimista, más probable y pesimista. La distribución triangular representa estos tres escenarios.

In [ ]:
componentes = pd.DataFrame({
    'componente': ['Ingeniería', 'Materiales', 'Mano de obra', 'Logística', 'Puesta en marcha'],
    'optimista': [45_000, 120_000, 80_000, 18_000, 25_000],
    'mas_probable': [55_000, 150_000, 100_000, 25_000, 35_000],
    'pesimista': [75_000, 210_000, 145_000, 42_000, 55_000],
})
presupuesto_autorizado = 430_000
componentes

## 2. Simulación de escenarios

Cada fila representa un escenario posible del proyecto. La semilla fija permite reproducir los resultados en Colab.

In [ ]:
escenarios = pd.DataFrame({
    fila.componente: rng.triangular(fila.optimista, fila.mas_probable, fila.pesimista, N_SIMULACIONES)
    for fila in componentes.itertuples(index=False)
})
escenarios['costo_total'] = escenarios.sum(axis=1)
escenarios.head()

In [ ]:
resumen = pd.Series({
    'Costo mínimo simulado': escenarios.costo_total.min(),
    'Promedio': escenarios.costo_total.mean(),
    'Desviación estándar': escenarios.costo_total.std(),
    'P50': escenarios.costo_total.quantile(0.50),
    'P80': escenarios.costo_total.quantile(0.80),
    'P90': escenarios.costo_total.quantile(0.90),
    'Costo máximo simulado': escenarios.costo_total.max(),
})
resumen

## 3. Probabilidad de cumplir el presupuesto

Se calcula la proporción de escenarios cuyo costo total no supera el presupuesto autorizado.

In [ ]:
probabilidad_cumplimiento = (escenarios.costo_total <= presupuesto_autorizado).mean()
sobrecosto_esperado = (escenarios.costo_total - presupuesto_autorizado).clip(lower=0).mean()
pd.Series({
    'Presupuesto autorizado': presupuesto_autorizado,
    'Probabilidad de cumplir': probabilidad_cumplimiento,
    'Sobrecosto promedio cuando ocurre': sobrecosto_esperado,
})

## 4. Visualización de resultados

La línea vertical muestra el presupuesto autorizado; el área a la derecha representa escenarios que lo exceden.

In [ ]:
plt.figure(figsize=(11, 5))
plt.hist(escenarios.costo_total, bins=70, color='#0f766e', alpha=0.82, edgecolor='white')
plt.axvline(presupuesto_autorizado, color='#b91c1c', linestyle='--', linewidth=2, label='Presupuesto autorizado')
plt.axvline(escenarios.costo_total.quantile(0.80), color='#d97706', linestyle=':', linewidth=2, label='P80')
plt.title('Distribución simulada del costo total')
plt.xlabel('Costo total')
plt.ylabel('Número de escenarios')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Interpretación para la decisión

- P50 representa un escenario central.
- P80 y P90 ofrecen referencias de planeación con mayor cobertura.
- Una probabilidad baja de cumplimiento indica la necesidad de revisar supuestos o incrementar la contingencia.
- La simulación no elimina la incertidumbre: la hace visible y cuantificable.

In [ ]:
recomendacion = pd.DataFrame({
    'métrica': ['P50', 'P80', 'P90', 'Probabilidad de cumplir'],
    'valor': [
        escenarios.costo_total.quantile(0.50),
        escenarios.costo_total.quantile(0.80),
        escenarios.costo_total.quantile(0.90),
        probabilidad_cumplimiento,
    ]
})
recomendacion

## Conclusión

La simulación Monte Carlo transforma estimaciones puntuales en un rango de resultados posibles. La reserva de contingencia debe seleccionarse con un nivel de cobertura acorde con el riesgo que la organización esté dispuesta a aceptar.